In [ ]:
# Power at one (Time, N, k) cell via repeated random draws + t-test
power_at_k <- function(data, time_val, n_val, k, n_iter = 1000) {
  sub <- data %>% filter(Time == time_val, N == n_val)
  control_vals <- sub %>% filter(Model == "control") %>% pull(Value)
  intervention_vals <- sub %>% filter(Model == "intervention") %>% pull(Value)

  sig <- replicate(n_iter, {
    x <- sample(control_vals, k, replace = FALSE)
    y <- sample(intervention_vals, k, replace = FALSE)
    t.test(x, y)$p.value < 0.05
  })

  mean(sig)
}

# Smallest k reaching power_threshold, scanning k_min:k_max
find_min_k <- function(data, time_val, n_val, power_threshold = 0.8,
                        k_min = 2, k_max = 100, n_iter = 1000) {
  for (k in k_min:k_max) {
    if (power_at_k(data, time_val, n_val, k, n_iter) >= power_threshold) {
      return(k)
    }
  }
  NA
}

# Compare each n against the n=50 baseline
compare_n_to_baseline <- function(data, time_val, n_values = c(10, 30, 50),
                                   baseline_n = 50, power_threshold = 0.8,
                                   k_min = 2, k_max = 100, n_iter = 1000) {

  min_k <- sapply(n_values, function(n_val) {
    find_min_k(data, time_val, n_val, power_threshold, k_min, k_max, n_iter)
  })

  baseline_k <- min_k[n_values == baseline_n]

  tibble(
    N = n_values,
    min_k_for_power_0.8 = min_k,
    extra_k_vs_n50 = min_k - baseline_k
  )
}

In [ ]:
library(tidyverse)

# 90% crash seasonal, n=50 baseline (original wrangle.R output: Time, Seed, Model, Value)
tajimasD_n50 <- read_csv("../data/Seasonal_0.1/hetero_0.5/Tajimas_D.csv", show_col_types = FALSE) %>%
  mutate(N = 50)

# subsampled n=10, 30 (Time, N, Seed, Model, Value)
tajimasD_subsample <- read_csv("../data/TajimasD_subsample.csv", show_col_types = FALSE)

# combine
data <- bind_rows(tajimasD_n50, tajimasD_subsample) %>%
  select(Time, N, Seed, Model, Value)

result <- compare_n_to_baseline(data, time_val = 6, n_values = c(10, 30, 50))
print(result)

In [ ]:
# 50% crash
tajimasD_n50 <- read_csv("../data/Seasonal_0.5/hetero_0.5/Tajimas_D.csv", show_col_types = FALSE) %>%
  mutate(N = 50)

tajimasD_subsample <- read_csv("../data/TajimasD_subsample_0.5.csv", show_col_types = FALSE)

data <- bind_rows(tajimasD_n50, tajimasD_subsample) %>%
  select(Time, N, Seed, Model, Value)

result <- compare_n_to_baseline(data, time_val = 6, n_values = c(10, 30, 50))
print(result)